In [5]:
pip install wikipedia tqdm biopython pandas

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tqdm (from versions: none)
ERROR: No matching distribution found for tqdm


In [3]:
import wikipedia
import requests
import pandas as pd
import time
import xml.etree.ElementTree as ET

# Wikipedia Fetcher (clean)
def get_wikipedia_summary(term):
    try:
        page = wikipedia.page(term, auto_suggest=False)
        title = page.title
        summary = wikipedia.summary(term, sentences=3)
        return title, summary
    except Exception as e:
        print(f"[Wikipedia] Error for '{term}': {e}")
        return None, None

# PubMed Abstract Fetcher using requests instead of Bio.Entrez
def get_pubmed_abstracts(term, max_results=1):
    try:
        # Search for IDs
        search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        search_params = {
            "db": "pubmed",
            "term": term,
            "retmax": max_results,
            "retmode": "json"
        }
        
        search_response = requests.get(search_url, params=search_params)
        search_data = search_response.json()
        
        ids = search_data.get("esearchresult", {}).get("idlist", [])
        if not ids:
            return None
            
        summaries = []
        for pmid in ids:
            # Fetch article details
            fetch_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
            fetch_params = {
                "db": "pubmed",
                "id": pmid,
                "rettype": "abstract",
                "retmode": "xml"
            }
            
            fetch_response = requests.get(fetch_url, params=fetch_params)
            
            # Parse XML
            root = ET.fromstring(fetch_response.content)
            
            # Extract title and abstract
            try:
                title_elem = root.find(".//ArticleTitle")
                abstract_elem = root.find(".//AbstractText")
                
                if title_elem is not None and abstract_elem is not None:
                    title = title_elem.text
                    abstract = abstract_elem.text
                    summaries.append(f"{title}:\n{abstract}")
            except Exception as e:
                print(f"Error parsing article {pmid}: {e}")
                
            time.sleep(0.3)
            
        return "\n\n".join(summaries)
    except Exception as e:
        print(f"[PubMed] Error for '{term}': {e}")
        return None

# Medical terms
medical_terms = [
    "Diabetes", "Hypertension", "Asthma", "Cancer (disease)", "Heart Disease"
]

# Output
data = []

for term in medical_terms:
    print(f"🔍 Fetching: {term}")
    wiki_title, wiki_summary = get_wikipedia_summary(term)
    pubmed_info = get_pubmed_abstracts(term)
    data.append({
        "term": term,
        "wikipedia_title": wiki_title,
        "wikipedia_summary": wiki_summary,
        "pubmed_article": pubmed_info
    })
    time.sleep(1)

# Save as CSV
df = pd.DataFrame(data)
df.to_csv("clean_medical_knowledge_base.csv", index=False)
print("✅ Clean knowledge base saved as 'clean_medical_knowledge_base.csv'")

ModuleNotFoundError: No module named 'wikipedia'